___
# <center>Associação entre uma variável numérica e uma categórica</center>
___

## Aula 17

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * comparar duas proporções, com o teste e com o intervalo de confiança da
   diferença, e ler o intervalo do jeito certo;
 * reconhecer quando uma variável exige leitura do texto e não cabe numa
   expressão regular;
 * especificar, com `pydantic`, os campos que um modelo de linguagem deve
   extrair de uma petição;
 * comparar uma variável numérica entre dois grupos com o teste t, e entre três
   ou mais com a ANOVA e o teste de Tukey;
 * escrever o resultado, inclusive quando os dados não mostram diferença.

___
<div id="indice"></div>

## Índice

- [Duas proporções, revisitando o qui-quadrado](#proporcoes)
- [Exercício de duas proporções](#exercicio_prop)
- [Variáveis que exigem leitura](#leitura)
- [Especificação dos campos com pydantic](#pydantic)
- [Teste t para dois grupos](#t)
- [ANOVA para três ou mais grupos](#anova)
- [Exercício](#exercicio)

> **Sobre a base.** Os dados são do projeto da clínica com o BTG e estão cobertos
> por sigilo bancário e pelo termo de parceria. O arquivo `btg_aulas.csv` está no
> BlackBoard e **não pode ser compartilhado** fora da turma, nem subido para o
> GitHub, nem colado em ferramentas externas além das usadas em aula.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_aulas.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
processos = pd.read_csv("btg_aulas.csv")
processos.shape

[Volta ao Índice](#indice)

___
<div id="proporcoes"></div>

# Duas proporções, revisitando o qui-quadrado

Na aula 16 testamos a independência entre duas categóricas com o qui-quadrado.
Quando **as duas são binárias**, dá para fazer a mesma pergunta de um jeito mais
direto: comparar duas proporções. A pergunta: **a proporção de pagamento é a
mesma nos processos com e sem contestação?**

In [ ]:
pd.crosstab(processos["tem_contestacao"], processos["pagou"], margins=True)

In [ ]:
# a proporção de pagamento dentro de cada grupo
pd.crosstab(processos["tem_contestacao"], processos["pagou"], normalize="index").round(3)

O banco pagou em 15,5% dos processos sem contestação e em 10,5% dos com
contestação. Uma diferença de 5 pontos. É grande o bastante para não ser acaso?

Primeiro, o que já sabemos fazer: o qui-quadrado da tabela 2x2.

In [ ]:
from scipy.stats import chi2_contingency

tabela = pd.crosstab(processos["tem_contestacao"], processos["pagou"])
qui, valor_p, gl, esperado = chi2_contingency(tabela, correction=False)
print(f"qui-quadrado = {qui:.2f}   gl = {gl}   valor-p = {valor_p:.4f}")

Agora o teste de duas proporções. Ele compara diretamente $\hat p_1$ (sem
contestação) e $\hat p_2$ (com contestação). Se H0 for verdadeira, as duas
proporções populacionais são iguais, e a melhor estimativa dessa proporção comum
é a dos dois grupos juntos, $\hat p$. A estatística é

$$z = \frac{\hat p_1 - \hat p_2}{\sqrt{\hat p\,(1 - \hat p)\left(\dfrac{1}{n_1} + \dfrac{1}{n_2}\right)}}$$

Com os números da tabela: $\hat p_1 = 27/174 = 0{,}155$, $\hat p_2 = 66/626 =
0{,}105$ e $\hat p = 93/800 = 0{,}116$.

$$z = \frac{0{,}155 - 0{,}105}{\sqrt{0{,}116 \times 0{,}884 \times \left(\dfrac{1}{174} + \dfrac{1}{626}\right)}} = \frac{0{,}050}{0{,}0275} \approx 1{,}81$$

In [ ]:
import numpy as np

sem = processos[~processos["tem_contestacao"]]["pagou"]
com = processos[processos["tem_contestacao"]]["pagou"]

# a conta à mão
p1, n1 = sem.mean(), len(sem)
p2, n2 = com.mean(), len(com)
p_comum = (sem.sum() + com.sum()) / (n1 + n2)

z_mao = (p1 - p2) / np.sqrt(p_comum * (1 - p_comum) * (1 / n1 + 1 / n2))
print(f"p1 = {p1:.3f}   p2 = {p2:.3f}   p comum = {p_comum:.3f}   z = {z_mao:.2f}")

In [ ]:
# a mesma conta, pronta no statsmodels
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

z, valor_p = proportions_ztest([sem.sum(), com.sum()], [len(sem), len(com)])
print(f"sem contestação: {sem.mean():.3f} (n = {len(sem)})   com contestação: {com.mean():.3f} (n = {len(com)})")
print(f"z = {z:.2f}   valor-p = {valor_p:.4f}")
print(f"z ao quadrado = {z**2:.2f}")

**O mesmo valor-p.** Numa tabela 2x2, sem a correção de continuidade, o
qui-quadrado é exatamente o $z$ ao quadrado: são o mesmo teste escrito de dois
jeitos. Então para que serve o de duas proporções? Para o que vem agora: o
**intervalo de confiança da diferença**, que diz de quanto é a diferença, e não
só se ela existe.

## O intervalo de confiança da diferença

No intervalo não supomos que H0 seja verdadeira, então cada proporção entra
com a sua própria variância, em vez da proporção comum:

$$EP = \sqrt{\frac{\hat p_1(1-\hat p_1)}{n_1} + \frac{\hat p_2(1-\hat p_2)}{n_2}}$$

e o intervalo de 95% é a diferença mais ou menos $1{,}96 \times EP$. A mesma
receita do intervalo para uma média, da aula 12.

In [ ]:
dif = p1 - p2
ep = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
print(f"diferença: {dif*100:.1f} pontos   erro-padrão: {ep*100:.1f} pontos")
print(f"IC 95%: de {(dif - 1.96*ep)*100:.1f} a {(dif + 1.96*ep)*100:.1f} pontos")

In [ ]:
# a mesma conta, pronta no statsmodels
baixo, alto = confint_proportions_2indep(sem.sum(), n1, com.sum(), n2, method="wald")
print(f"IC 95%: de {baixo*100:.1f} a {alto*100:.1f} pontos")

## O que quer dizer "95% de confiança"

A diferença verdadeira, na população de todos os processos, é um número fixo que
não conhecemos. O que varia é a **amostra**, e com ela o intervalo. Para ver isso,
vamos fingir que conhecemos a verdade: as proporções da nossa amostra, 15,5% e
10,5%. Sorteamos 40 amostras novas, do mesmo tamanho, e calculamos o intervalo
em cada uma.

In [ ]:
rng = np.random.default_rng(2026)
verdade = (p1 - p2) * 100


def sorteia_intervalos(quantos):
    a = rng.binomial(n1, p1, quantos) / n1
    b = rng.binomial(n2, p2, quantos) / n2
    d = (a - b) * 100
    m = 1.96 * np.sqrt(a * (1 - a) / n1 + b * (1 - b) / n2) * 100
    return pd.DataFrame({"amostra": np.arange(1, quantos + 1), "dif": d,
                         "baixo": d - m, "alto": d + m})


intervalos = sorteia_intervalos(40)
intervalos["contém a verdade"] = np.where(
    (intervalos["baixo"] <= verdade) & (verdade <= intervalos["alto"]), "sim", "não")
intervalos.head()

In [ ]:
from plotnine import (ggplot, aes, geom_segment, geom_point, geom_vline, labs,
                      scale_color_manual, theme_minimal)

(ggplot(intervalos, aes(y="amostra", color="contém a verdade"))
 + geom_vline(xintercept=0, color="gray")
 + geom_vline(xintercept=verdade, linetype="dashed", color="#1f6fb4")
 + geom_segment(aes(x="baixo", xend="alto", yend="amostra"), size=1)
 + geom_point(aes(x="dif"))
 + scale_color_manual(values={"sim": "#1f6fb4", "não": "#e50505"})
 + labs(x="diferença em pontos percentuais", y="amostra sorteada")
 + theme_minimal())

Rode a célula de sorteio de novo e veja o gráfico mudar. Em cada rodada, quase
todos os intervalos cruzam a linha tracejada; de vez em quando um fica de fora,
em vermelho. Com muitas repetições, a proporção que acerta chega a 95%:

In [ ]:
muitos = sorteia_intervalos(10_000)
((muitos["baixo"] <= verdade) & (verdade <= muitos["alto"])).mean()

**A confiança é no método, não no intervalo.** Um intervalo específico, como o
nosso de -0,9 a 10,9, contém ou não contém a diferença verdadeira; não há 95% de
probabilidade nele. O que tem 95% é a receita: usada muitas vezes, ela acerta 95%
das vezes.

## Como relatar um valor-p de 0,07

**Não escreva:** "não há diferença entre processos com e sem contestação".

**Escreva:** a diferença estimada é de **5 pontos percentuais**, com intervalo de
95% de -0,9 a 10,9 pontos (valor-p = 0,07). **Estes dados não mostram
diferença** ao nível de 5%, mas são compatíveis tanto com nenhuma diferença
quanto com uma diferença de 10 pontos.

Um valor-p de 0,07 e um de 0,04 descrevem evidências parecidas. Relatar a
estimativa e o intervalo evita transformar uma régua contínua num carimbo de sim
ou não.

[Volta ao Índice](#indice)

___
<div id="exercicio_prop"></div>

# Exercício de duas proporções

Em duplas. **A proporção de pagamento é a mesma nos processos com e sem petição
inicial entre os documentos que o banco guardou?** Complete as lacunas `____` e
escreva o resultado em uma frase, com a diferença, o intervalo e o valor-p.

In [ ]:
com = processos[processos["tem_inicial"]]["pagou"]
sem = processos[~processos["____"]]["pagou"]
print(f"com inicial: {com.mean():.3f} (n = {len(com)})   sem inicial: {sem.mean():.3f} (n = {len(sem)})")

In [ ]:
z, valor_p = proportions_ztest([sem.sum(), com.sum()], [len(sem), len(____)])
baixo, alto = confint_proportions_2indep(sem.sum(), len(sem), com.sum(), len(com), method="wald")
print(f"z = {z:.2f}   valor-p = {valor_p:.6f}")
print(f"diferença: de {baixo*100:.1f} a {alto*100:.1f} pontos percentuais (IC 95%)")

**Sua frase:** ____

Cuidado: o teste mostra associação. Não escreva que ter a inicial no dossiê
**causa** alguma coisa.

[Volta ao Índice](#indice)

___
<div id="leitura"></div>

# Variáveis que exigem leitura

Na aula 16, o tribunal saiu do número CNJ com uma expressão regular, porque o
número tem sempre a mesma forma. Agora a pergunta é outra: **o que o autor está
pedindo?** Leia a seção de pedidos de uma inicial.

In [ ]:
print(processos.loc[processos["tem_inicial"], "texto_pedidos"].dropna().iloc[3][:1500])

"Revisão dos juros", "afastamento da tarifa de cadastro", "devolução em dobro":
cada advogado escreve de um jeito, em ordem diferente, com palavras diferentes.
Não há padrão de caracteres que capture isso com segurança. A informação está no
**sentido** do texto, e é aí que entra o modelo de linguagem.

[Volta ao Índice](#indice)

___
<div id="pydantic"></div>

# Especificação dos campos com pydantic

Antes de chamar o modelo, declaramos a **forma da resposta**: quais campos, de
que tipo, e com que valores possíveis. O `Literal` fecha as categorias, e a
`description` de cada campo é a pergunta que o modelo recebe.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Tese = Literal["juros abusivos", "tarifas e encargos", "seguro embutido",
               "capitalização de juros", "outra"]


class Inicial(BaseModel):
    tese_principal: Tese = Field(
        description="O principal ponto do contrato que o autor ataca. Se houver "
                    "mais de um, o que ocupa mais espaço na petição.")
    pede_dano_moral: bool = Field(
        description="O autor pede indenização por dano moral?")
    qtd_pedidos: int = Field(
        ge=1, le=15,
        description="Quantos pedidos de mérito distintos o autor faz, sem contar "
                    "gratuidade, citação e honorários.")

A chamada ao modelo cabe em poucas linhas. Ela **não roda na aula**, porque exige
uma chave de acesso; as colunas que ela produziu já estão na base. O código
completo, que leu as 484 petições iniciais da amostra, está em
`notebooks/estruturar.py`.

```python
from pydantic_ai import Agent

agente = Agent("openai:gpt-6-luna", output_type=Inicial,
               system_prompt="Você lê petições iniciais contra o Banco PAN ...")
resposta = agente.run_sync(texto_da_peticao).output
resposta.tese_principal, resposta.qtd_pedidos
```

In [ ]:
iniciais = processos[processos["tem_inicial"]].copy()
iniciais["tese_principal"].value_counts()

In [ ]:
# proporção de petições com cada pedido

(
    iniciais
    [["pede_dano_moral", "pede_tarifas", "pede_seguro", "pede_repeticao_dobro"]]
    .eq(True)
    .mean()
    .round(3)
)

[Volta ao Índice](#indice)

___
<div id="t"></div>

# Teste t para dois grupos

Pergunta: **o valor financiado é o mesmo nos processos em que o banco pagou e
nos outros?** Uma numérica e uma binária: o gráfico é o boxplot por grupo.

In [ ]:
from plotnine import ggplot, aes, geom_boxplot, labs, theme_minimal, coord_flip

(
    ggplot(processos.dropna(subset=["valor_financiado"]), aes(x="pagou", y="valor_financiado"))
    + geom_boxplot(fill="#cde2fb")
    + labs(x="o banco pagou?", y="valor financiado (R$)")
    + theme_minimal()
    + coord_flip()
)

In [ ]:
processos.groupby("pagou")["valor_financiado"].describe().round(0)

In [ ]:
from scipy.stats import ttest_ind

pagou = processos.loc[processos["pagou"], "valor_financiado"].dropna()
nao_pagou = processos.loc[~processos["pagou"], "valor_financiado"].dropna()

t, valor_p = ttest_ind(pagou, nao_pagou, equal_var=False)
print(f"com pagamento: R$ {pagou.mean():,.0f} (n = {len(pagou)})")
print(f"sem pagamento: R$ {nao_pagou.mean():,.0f} (n = {len(nao_pagou)})")
print(f"t = {t:.2f}   valor-p = {valor_p:.6f}")

In [ ]:
import numpy as np
from scipy.stats import t as dist_t

dif = pagou.mean() - nao_pagou.mean()
ep = np.sqrt(pagou.var() / len(pagou) + nao_pagou.var() / len(nao_pagou))
margem = dist_t.ppf(0.975, df=min(len(pagou), len(nao_pagou)) - 1) * ep
print(f"diferença: R$ {dif:,.0f}, de R$ {dif - margem:,.0f} a R$ {dif + margem:,.0f} (IC 95%)")

**Como relatar:** nos processos em que o banco pagou, o valor financiado médio é
cerca de **R$ 6 mil menor** que nos demais (t = -4,76; valor-p < 0,001). Uma
explicação plausível é que acordo em contrato pequeno custa pouco. Plausível não
é demonstrado: o teste mostra associação, não o motivo.

[Volta ao Índice](#indice)

___
<div id="anova"></div>

# ANOVA para três ou mais grupos

Pergunta: **a quantidade de pedidos é a mesma nas diferentes teses?** A tese
principal e a quantidade de pedidos foram construídas pelo modelo. Com quatro
grupos, fazer vários testes t aumenta a chance de achar uma diferença por acaso;
a ANOVA faz uma pergunta só: alguma média é diferente das outras?

In [ ]:
frequentes = iniciais["tese_principal"].value_counts()
base = iniciais[iniciais["tese_principal"].isin(frequentes[frequentes >= 15].index)]
base = base.dropna(subset=["qtd_pedidos"])

(
    ggplot(base, aes(x="tese_principal", y="qtd_pedidos"))
    + geom_boxplot(fill="#cde2fb")
    + labs(x="", y="quantidade de pedidos")
    + theme_minimal()
    + coord_flip()
)

In [ ]:
from scipy.stats import f_oneway

grupos = [g["qtd_pedidos"].to_numpy() for _, g in base.groupby("tese_principal")]
F, valor_p = f_oneway(*grupos)
print(base.groupby("tese_principal")["qtd_pedidos"].agg(["mean", "count"]).round(2))
print(f"\nF = {F:.2f}   valor-p = {valor_p:.5f}")

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print(pairwise_tukeyhsd(base["qtd_pedidos"], base["tese_principal"]))

**Como relatar:** a quantidade de pedidos difere entre as teses (F = 7,26;
valor-p < 0,001). O Tukey mostra que a diferença está no grupo "outra", com cerca
de 3 pedidos em média contra 5 a 6 nos demais; entre juros, capitalização e
tarifas os dados não mostram diferença.

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# Exercício

Em duplas. Complete as lacunas `____` e rode cada célula. Em cada item: o
gráfico, o teste e uma frase.

## Item 1: valor da parcela e pagamento

**O valor da parcela é o mesmo nos processos com e sem pagamento?**

In [ ]:
# 1a. o gráfico: o boxplot da parcela em cada grupo
(
    ggplot(processos.dropna(subset=["valor_parcela"]), aes(x="pagou", y="____"))
    + geom_boxplot(fill="#cde2fb")
    + labs(x="o banco pagou?", y="valor da parcela (R$)")
    + theme_minimal()
    + coord_flip()
)

In [ ]:
# 1b. o teste: são dois grupos, então teste t
com_pag = processos.loc[processos["pagou"], "valor_parcela"].dropna()
sem_pag = processos.loc[~processos["____"], "valor_parcela"].dropna()

t, valor_p = ____(com_pag, sem_pag, equal_var=False)
print(f"com pagamento: R$ {com_pag.mean():,.0f}   sem pagamento: R$ {sem_pag.mean():,.0f}")
print(f"t = {t:.2f}   valor-p = {valor_p:.6f}")

**Sua frase:** ____

## Item 2: taxa de juros e tese principal

**A taxa de juros do contrato é a mesma nas diferentes teses principais?** A
base `base`, da seção da ANOVA, já tem só as teses com pelo menos 15 petições.

In [ ]:
# 2a. o gráfico: o boxplot da taxa em cada tese
(
    ggplot(base.dropna(subset=["taxa_contrato_pct"]), aes(x="tese_principal", y="____"))
    + geom_boxplot(fill="#cde2fb")
    + labs(x="", y="taxa de juros do contrato (% ao mês)")
    + theme_minimal()
    + coord_flip()
)

In [ ]:
# 2b. o teste: são quatro grupos, então ANOVA
grupos = [g["taxa_contrato_pct"].dropna().to_numpy() for _, g in base.groupby("____")]
F, valor_p = ____(*grupos)

print(base.groupby("tese_principal")["taxa_contrato_pct"].mean().round(2))
print(f"F = {F:.2f}   valor-p = {valor_p:.3f}")

**Precisa do Tukey?** Lembre a regra: se a ANOVA não mostrar diferença, pare aí.

**Sua frase:** ____

[Volta ao Índice](#indice)